# Day 02 — Ingestion, parsing and metadata: messy files in, trustworthy records out

*Part of **15 Days of Modern, Enterprise-Relevant RAG from Scratch**. Lesson: [`docs/day-02/LEARN.md`](../docs/day-02/LEARN.md).*

On Day 01 every document was a tidy Markdown file. Real knowledge bases are not. Today the same folder also holds an intranet **HTML** page, a **PDF** with a table, a **scanned** PDF, and two **plain-text** files. Before anything can be retrieved, an **ingestion pipeline** has to turn all of them into one kind of record that later stages can trust.

```
 13 files ──> parse ──> clean ──> validate + attach permissions ──> 11 Document records  (outputs/day-02/documents.jsonl)
 (md, html,   one adapter   headers, footers,   from the system of record   └──> 2 quarantined, each with a reason
  pdf, txt)   per format    hyphens, tables
```

| This notebook **does** | This notebook **does not** |
|---|---|
| Parse Markdown, HTML, PDF and plain text into one `Document` schema | Chunk documents (Day 03) or build an index (Days 04–05) |
| Keep descriptive, access and provenance metadata for every record | Run OCR on scanned pages (it detects them and explains why) |
| Show where PDF extraction breaks: tables, headers, hyphens | Enforce permissions at query time (Days 05 and 14) |
| Quarantine files it can't trust, with a reason for each | Call any LLM or paid API |
| Detect added, changed, re-permissioned and deleted documents | Claim to be production-ready |

**Cost:** free. **Runtime:** a few seconds. **Requirements:** Python 3.11+, Jupyter and `pypdf` (all pinned in `requirements.txt`).
**How to run:** open this notebook from inside the cloned repository, then *Kernel → Restart Kernel and Run All Cells*.
**Verified:** executed top to bottom on 2026-10-13 on Linux with Python 3.11, 3.12, 3.13 and 3.14 (pypdf 6.20.1). The saved outputs come from the Python 3.13 run. CI runs the same notebook on Linux, Windows and macOS on every push.

## 0. Setup

One third-party package today: [`pypdf`](https://pypdf.readthedocs.io/), a pure-Python PDF library (BSD licence, no system dependencies). Everything else is the Python standard library, including the HTML parser.

The three settings in the second cell are explained where they are used.

In [1]:
import hashlib
import json
import platform
import re
import shutil
import sys
import tempfile
import unicodedata
from dataclasses import asdict, dataclass
from datetime import date
from html.parser import HTMLParser
from pathlib import Path

import pypdf
from pypdf import PdfReader

assert sys.version_info >= (3, 11), "This series needs Python 3.11 or newer."
print(f"Python {platform.python_version()} | pypdf {pypdf.__version__}")

Python 3.13.16 | pypdf 6.20.1


In [2]:
SNAPSHOT_DATE = date(2026, 10, 13)  # when this sample export was taken; review ages are measured from here
MAX_REVIEW_AGE_DAYS = 365           # an older review date gets a "review overdue" warning (section 5)
PDF_MODE = "layout"                 # pypdf extraction mode; section 3.3 shows why not the default "plain"

## 1. Inventory — what actually arrived?

Ingestion starts with an honest look at the inputs. Imagine this folder is a nightly export from Acme's document system: the eight Markdown policies from Day 01, plus five files in other formats.

Permissions are **not** in the folder. They come from a separate export, `data/sample/access-control.json`, which simulates the document system's sharing settings (the *system of record* for who may read what). Section 5 explains why that separation matters.

In [3]:
def find_repo_root(start: Path | None = None) -> Path:
    # Walk up until data/sample/acme-handbook is found, so the notebook works from the repo root
    # or from notebooks/. No hard-coded paths.
    start = (start or Path.cwd()).resolve()
    for folder in (start, *start.parents):
        if (folder / "data" / "sample" / "acme-handbook").is_dir():
            return folder
    raise FileNotFoundError(
        "Could not find data/sample/acme-handbook. Start Jupyter from inside the cloned repository."
    )


def list_files(folder: Path) -> list[Path]:
    # Sorted for the same order on every OS; hidden files such as .DS_Store are skipped.
    return sorted(p for p in folder.iterdir() if p.is_file() and not p.name.startswith("."))


REPO_ROOT = find_repo_root()
CORPUS_DIR = REPO_ROOT / "data" / "sample" / "acme-handbook"
files = list_files(CORPUS_DIR)

print(f"{len(files)} files in {CORPUS_DIR.relative_to(REPO_ROOT).as_posix()}/\n")
print(f"{'file':<34}{'format':<8}{'bytes':>7}")
for path in files:
    print(f"{path.name:<34}{path.suffix[1:]:<8}{path.stat().st_size:>7,}")

13 files in data/sample/acme-handbook/

file                              format    bytes
data-retention-policy.md          md          781
draft-reorg-announcement.txt      txt         342
expense-reimbursement-policy.md   md          949
it-software-requests.html         html      2,380
manager-pay-review-guide.txt      txt         480
new-starter-it-checklist.md       md          677
parental-leave-policy.md          md          741
password-and-mfa-standard.md      md          945
remote-work-policy.md             md        1,235
scanned-visitor-policy.pdf        pdf       8,789
security-incident-reporting.md    md          916
travel-policy.pdf                 pdf       4,818
vpn-access-guide.md               md          834


## 2. The contract — one record for every format

Downstream stages (chunking, indexing, retrieval, citations, permissions) should never need to know whether text came from a PDF or a web page. So ingestion's job is to fill **one schema** for every source. Its fields come from three places, and *where a field comes from decides how far you can trust it*:

| Group | Fields | Comes from | Trust |
|---|---|---|---|
| **Descriptive** | `title`, `owner`, `last_reviewed` | the document itself (front matter, `<meta>` tags, PDF properties) | Helpful, but any author can edit it, and it is often missing or stale. Validate it. |
| **Access** | `allowed_groups` | the **system of record** (the permission export) | Authoritative. **Never** read from the document's own text. |
| **Provenance** | `doc_id`, `source_path`, `source_format`, `parser`, `content_hash` | added by the pipeline | Lets you trace every answer back to a file, a parser version and an exact version of the text. |

Files the pipeline can't trust become `Quarantined` records with reasons. Nothing is dropped silently.

In [4]:
@dataclass(frozen=True)
class Document:
    # Identity
    doc_id: str                       # stable: the same file gets the same ID on every run
    # Descriptive metadata, read from the document (useful, but any author can edit it)
    title: str
    owner: str | None
    last_reviewed: str | None         # ISO date "YYYY-MM-DD", or None if missing or unreadable
    # Access metadata, from the system of record, never from the document's own text
    allowed_groups: tuple[str, ...]
    # Provenance, added by the pipeline
    source_path: str                  # relative to the repository root
    source_format: str                # "md", "html", "pdf" or "txt"
    parser: str                       # which code and library version produced `body`
    content_hash: str                 # SHA-256 of `body`: identifies this *version* of the text
    # Content: normalised text with "## " section headings, whatever the source format
    body: str
    warnings: tuple[str, ...] = ()    # problems worth fixing that don't block indexing


@dataclass(frozen=True)
class Quarantined:
    # A file the pipeline refused to index, and why. Reported, never silently dropped.
    source_path: str
    reasons: tuple[str, ...]


@dataclass
class Parsed:
    # What each format adapter returns: raw descriptive metadata plus body text.
    title: str | None
    owner: str | None
    last_reviewed: str | None
    body: str
    parser: str
    notes: tuple[str, ...] = ()       # adapter observations, e.g. "1 image, no text layer"

## 3. Parse — one adapter per format

Every format hides text and metadata in different places, so each gets a small **adapter** that returns a `Parsed` result. All adapters aim for the same body convention: plain text with `## ` section headings, which Day 03's chunker can split whatever the source was.

First, a shared **normaliser**. Text from different tools differs in invisible ways (line endings, non-breaking spaces, typographic ligatures), and those differences break exact matching and hashing. Unicode normalisation form NFKC folds many of them together.

In [5]:
def normalise(text: str) -> str:
    # Make text from every format comparable: one Unicode form, one line ending, tidy spaces.
    text = unicodedata.normalize("NFKC", text)        # e.g. ligature "ﬁ" -> "fi", no-break space -> space
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = [" ".join(line.split()) for line in text.split("\n")]
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


def space_headings(text: str) -> str:
    # One blank line before every heading, as in the Markdown sources.
    return re.sub(r"\n+(#{2,3} )", r"\n\n\1", text)


sample = "Ofﬁce hours:\r\n\r\n\r\n\r\n  10:00   to 15:00  "
print(repr(sample))
print(repr(normalise(sample)))

'Ofﬁce\xa0hours:\r\n\r\n\r\n\r\n  10:00   to 15:00  '
'Office hours:\n\n10:00 to 15:00'


NFKC is not free of side effects: it also turns `²` into `2` and `½` into `1⁄2`. For policies that's fine. For chemistry or maths documents you might choose the gentler NFC form. Decide once, apply everywhere, and record it, because the hash in section 4 depends on it.

### 3.1 Markdown — front matter (the Day 01 way)

Markdown files carry their descriptive metadata in a front-matter block. This is the Day 01 parser, unchanged, so Day 01's document IDs and citations stay valid.

In [6]:
def parse_front_matter(text: str) -> tuple[dict[str, str], str]:
    # Split a '---'-delimited 'key: value' header from the Markdown body.
    lines = text.splitlines()
    if not lines or lines[0].strip() != "---":
        return {}, text
    metadata: dict[str, str] = {}
    for i, line in enumerate(lines[1:], start=1):
        if line.strip() == "---":
            return metadata, "\n".join(lines[i + 1:]).strip()
        key, sep, value = line.partition(":")
        if sep:
            metadata[key.strip()] = value.strip()
    raise ValueError("Front matter opened with '---' but never closed.")


def parse_markdown(path: Path) -> Parsed:
    metadata, body = parse_front_matter(path.read_text(encoding="utf-8"))
    return Parsed(metadata.get("title"), metadata.get("owner"), metadata.get("last_reviewed"),
                  body, parser="front matter + Markdown (standard library)")


md_example = parse_markdown(CORPUS_DIR / "remote-work-policy.md")
print(f"title: {md_example.title} | owner: {md_example.owner} | last_reviewed: {md_example.last_reviewed}")
print("\n".join(md_example.body.splitlines()[:2]))

title: Remote Work Policy | owner: People Team | last_reviewed: 2026-03-02
## Purpose
This policy explains how Acme employees can work remotely while staying productive, connected and secure. It applies to all full-time and part-time employees. Contractors follow the terms in their contract.


### 3.2 HTML — keep the content, drop the furniture

An intranet page is mostly **boilerplate**: navigation, cookie banners, footers, scripts and styles. Indexed as-is, every page would match "privacy", "cookies" and "search", and every chunk would carry noise.

The rule here is simple and common: keep the text inside `<main>`, skip `<script>`, `<style>`, `<nav>`, `<header>`, `<footer>` and similar tags, and translate structure into the body convention. HTML *says* what is a heading, a list item or a table cell, so structure survives:

- `<h1>` → the title; `<h2>` → `## ` heading
- `<li>` → `1. ` or `- `
- each table row → `cell | cell | cell`

We use Python's built-in `html.parser` rather than a third-party library. The parser records everything it dropped, so you can check the rule didn't remove real content.

In [7]:
SKIP_TAGS = {"script", "style", "nav", "header", "footer", "aside", "noscript", "button", "form"}
BLOCK_TAGS = {"h1", "h2", "h3", "p", "li", "ol", "ul", "table", "tr", "td", "th", "br", "div", "section"}


class PageTextParser(HTMLParser):
    # Keeps readable text inside <main> as Markdown-style lines and reads <meta name=... content=...>.
    # Text outside <main>, or inside SKIP_TAGS, goes to `dropped` so you can audit what was removed.

    def __init__(self) -> None:
        super().__init__(convert_charrefs=True)       # "&copy;" -> "©"
        self.meta: dict[str, str] = {}
        self.title_tag, self.h1 = "", ""
        self.lines: list[str] = []
        self.dropped: list[str] = []
        self._text: list[str] = []                    # text of the block being read
        self._cells: list[str] | None = None          # cells of the table row being read
        self._lists: list[list] = []                  # [kind, counter] for each open <ol>/<ul>
        self._in_title = False
        self._main_depth = self._skip_depth = 0

    def _keeping(self) -> bool:
        return self._main_depth > 0 and self._skip_depth == 0

    def _take_text(self) -> str:
        text, self._text = " ".join("".join(self._text).split()), []
        return text

    def _flush(self, prefix: str = "") -> None:
        if text := self._take_text():
            self.lines.append(prefix + text)

    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        if tag == "meta" and attrs.get("name"):
            self.meta[attrs["name"].lower()] = (attrs.get("content") or "").strip()
        elif tag == "title":
            self._in_title = True
        elif tag == "main":
            self._main_depth += 1
        elif tag in SKIP_TAGS:
            self._skip_depth += 1
        elif tag in ("ol", "ul"):
            self._lists.append([tag, 0])
        elif tag == "li" and self._lists:
            self._lists[-1][1] += 1
        elif tag == "tr":
            self._cells = []
        if tag in BLOCK_TAGS and self._keeping():
            self._flush()                             # loose text before a block becomes its own line

    def handle_endtag(self, tag):
        if tag == "title":
            self._in_title = False
        elif tag == "main":
            self._main_depth -= 1
        elif tag in SKIP_TAGS:
            self._skip_depth -= 1
        if not self._keeping():
            return
        if tag == "h1":
            self.h1 = self._take_text()
        elif tag in ("h2", "h3"):
            self._flush("## " if tag == "h2" else "### ")
        elif tag == "li":
            kind, n = self._lists[-1] if self._lists else ("ul", 0)
            self._flush(f"{n}. " if kind == "ol" else "- ")
        elif tag in ("td", "th") and self._cells is not None:
            self._cells.append(self._take_text())
        elif tag == "tr" and self._cells is not None:
            self.lines.append(" | ".join(self._cells))
            self._cells = None
        elif tag in ("ol", "ul") and self._lists:
            self._lists.pop()
        elif tag == "p":
            self._flush()

    def handle_data(self, data):
        if self._in_title:
            self.title_tag += data
        elif self._keeping():
            self._text.append(data)
        elif data.strip():
            self.dropped.append(" ".join(data.split()))


def read_html(path: Path) -> PageTextParser:
    page = PageTextParser()
    page.feed(path.read_text(encoding="utf-8"))
    page.close()
    return page


def parse_html(path: Path) -> Parsed:
    page = read_html(path)
    return Parsed(title=page.h1 or page.title_tag.strip() or None,   # <title> often carries a site suffix
                  owner=page.meta.get("owner"), last_reviewed=page.meta.get("last-reviewed"),
                  body="\n".join(page.lines), parser="html.parser, <main> only (standard library)")


page = read_html(CORPUS_DIR / "it-software-requests.html")
print(f"<title>: {page.title_tag!r}   <h1>: {page.h1!r}   <meta>: {page.meta}\n")
print("KEPT")
print("\n".join(page.lines))
print("\nDROPPED")
for text in page.dropped:
    print("  x", text[:90])

<title>: 'Software requests | Acme Intranet'   <h1>: 'Software requests'   <meta>: {'owner': 'IT Service Desk', 'last-reviewed': '2025-06-30'}

KEPT
Request any software you need for work through the self-service portal. Do not install software from the internet yourself: unapproved software is a common way for malware to reach company laptops.
## How to request software
1. Search the software catalogue in the self-service portal.
2. If the software is listed, select Install. Catalogue software installs automatically.
3. If it is not listed, submit a new software request and explain the business need.
## Approval times
Request type | Who approves | Typical time
Catalogue software | No approval needed | Same day
Paid licence | Your manager | 3 business days
Browser extension | IT Security | 5 business days
Admin rights | IT Security and your manager | 5 business days
## Free and open-source software
Free software still needs a request: licences and security updates matter even when ther

Two details worth noticing:

- The `<title>` says *"Software requests | Acme Intranet"*. The site suffix is navigation, not content, so the adapter prefers the page's `<h1>`.
- The table survived as rows: `Paid licence | Your manager | 3 business days`. HTML marks up what a row and a cell are. PDF, next, does not.

### 3.3 PDF — text extraction is reconstruction

A PDF is a set of **drawing instructions**: "put these glyphs at these coordinates". It has no built-in notion of a paragraph, a heading or a table row, so every extractor has to *reconstruct* reading order from positions. Our sample PDF is synthetic and generated by [`scripts/data/make_sample_pdfs.py`](../scripts/data/make_sample_pdfs.py), with three common quirks built in on purpose. Here is pypdf's default (`"plain"`) extraction of page 1:

In [8]:
TRAVEL_PDF = CORPUS_DIR / "travel-policy.pdf"
reader = PdfReader(TRAVEL_PDF)
print(f"{len(reader.pages)} pages | document properties: {dict(reader.metadata)}\n")
print(reader.pages[0].extract_text())

2 pages | document properties: {'/Title': 'Travel Policy', '/Owner': 'Finance', '/LastReviewed': '2026-04-15', '/Producer': 'scripts/data/make_sample_pdfs.py (synthetic sample)'}

Acme Corp  |  Travel Policy  |  Internal
Acme Corp - internal use only - Page 1 of 2
Travel Policy
1. Purpose
This policy explains how Acme employees book business travel and claim reim-
bursement for travel costs. It applies to all employees, and to contractors who
travel at Acme's request.
2. Booking travel
Book flights, trains and hotels through the travel portal at least 14 days before
you travel. Economy class is standard for flights under six hours. Flights of six
hours or more may be booked in premium economy with your manager's approval.
3. Daily limits by city
Hotel and meal costs are reimbursed up to the daily limits below. Limits are per
person and include taxes. Use the Other cities row for any city not listed.
City
London
New York
Singapore
Other cities
Hotel per night
USD 250
USD 300
USD 220
USD

Read it as a machine would:

1. **The footer comes second.** *"Acme Corp - internal use only - Page 1 of 2"* appears right after the header because the generator drew both before the body. Running headers and footers repeat on every page and pollute every chunk.
2. **"reim-" / "bursement".** A word hyphenated at a line break is now two non-words. A keyword search for *reimbursement* will not find this sentence.
3. **The table is gone.** The generator drew the table one column at a time, and plain extraction follows the drawing order: all cities, then all hotel limits, then all meal limits. Which number belongs to Singapore? The text no longer says.

pypdf's **layout mode** places text by its position on the page instead, which keeps each table row on one line:

In [9]:
layout_text = reader.pages[0].extract_text(extraction_mode="layout", layout_mode_space_vertically=False)
for line in layout_text.splitlines():
    if "USD" in line or "City" in line:
        print(line.rstrip())

  City                                     Hotel per night                        Meals per day
  London                                   USD 250                                USD 75
  New York                                 USD 300                                USD 75
  Singapore                                USD 220                                USD 60
  Other cities                             USD 180                                USD 50


Layout mode is not magic either: it pads columns with spaces, so turning those gaps back into cells is a heuristic (here, 3+ spaces = a column break). It works for this clean table. Merged cells, wrapped cell text or multi-column pages need a table-aware tool or a structured source; section 7 and `LEARN.md` cover the options.

The PDF adapter cleans the extracted lines in four steps, each one undoing a quirk above:

1. **Drop running lines**: lines that appear on *every* page once digits are masked (`Page 1 of 2` → `Page # of #`).
2. **Rebuild table rows**: split on wide gaps and join the cells with ` | `, like the HTML adapter.
3. **Rebuild headings**: a short numbered line such as `1. Purpose` becomes `## Purpose`.
4. **Re-join lines**: undo end-of-line hyphenation (`reim-` + `bursement`) and join the wrapped lines of a paragraph.

It also checks whether the PDF has any text at all, which leads to the scanned file in a moment.

In [10]:
TABLE_GAP = re.compile(r"\s{3,}")                         # layout mode pads table columns with spaces
NUMBERED_HEADING = re.compile(r"^\d{1,2}\.\s+([A-Z][^.]{0,60})$")   # "3. Daily limits by city"


def pdf_page_lines(reader: PdfReader, mode: str) -> list[list[str]]:
    # Non-empty text lines, one list per page.
    pages = []
    for page in reader.pages:
        if mode == "layout":
            text = page.extract_text(extraction_mode="layout", layout_mode_space_vertically=False)
        else:
            text = page.extract_text()
        pages.append([line.rstrip() for line in text.splitlines() if line.strip()])
    return pages


def remove_running_lines(pages: list[list[str]]) -> tuple[list[list[str]], set[str]]:
    # Drop lines that appear on every page once digits are masked: headers, footers, page numbers.
    if len(pages) < 2:
        return pages, set()

    def mask(line: str) -> str:
        return re.sub(r"\d+", "#", " ".join(line.split()))

    on_every_page = set.intersection(*({mask(line) for line in page} for page in pages))
    return [[line for line in page if mask(line) not in on_every_page] for page in pages], on_every_page


def pdf_lines_to_body(lines: list[str], title: str | None) -> str:
    out: list[str] = []
    for line in (raw.strip() for raw in lines):
        previous = out[-1] if out else ""
        cells = TABLE_GAP.split(line)
        if not out and line == title:
            continue                                          # the big title line is metadata, not body
        if len(cells) > 1:
            out.append(" | ".join(cells))                     # 2. a table row
        elif (heading := NUMBERED_HEADING.match(line)):
            out.append(f"## {heading.group(1)}")              # 3. "1. Purpose" -> "## Purpose"
        elif previous.endswith("-") and previous[-2:-1].isalpha() and line[:1].islower():
            out[-1] = previous[:-1] + line                    # 4. "reim-" + "bursement"
        elif previous and not previous.startswith("#") and " | " not in previous:
            out[-1] = previous + " " + line                   # 4. same paragraph, next visual line
        else:
            out.append(line)
    return "\n".join(out)


def parse_pdf(path: Path, mode: str | None = None) -> Parsed:
    mode = mode or PDF_MODE
    reader = PdfReader(path)
    info = reader.metadata or {}
    title, parser = info.get("/Title"), f"pypdf {pypdf.__version__} ({mode} mode)"
    pages = pdf_page_lines(reader, mode)
    if not any(pages):                                        # no text layer at all
        images = sum(len(page.images) for page in reader.pages)
        note = f"{len(reader.pages)} page, {images} image, 0 characters of text"
        return Parsed(title, info.get("/Owner"), info.get("/LastReviewed"), "", parser, (note,))
    pages, running = remove_running_lines(pages)              # 1. headers and footers
    body = pdf_lines_to_body([line for page in pages for line in page], title)
    return Parsed(title, info.get("/Owner"), info.get("/LastReviewed"), body, parser,
                  (f"removed running lines: {sorted(running)}",) if running else ())


travel = parse_pdf(TRAVEL_PDF)
print(f"title: {travel.title} | owner: {travel.owner} | last_reviewed: {travel.last_reviewed}")
print(f"parser: {travel.parser}")
print(f"notes: {travel.notes}\n")
print(space_headings(normalise(travel.body)))

title: Travel Policy | owner: Finance | last_reviewed: 2026-04-15
parser: pypdf 6.20.1 (layout mode)
notes: ("removed running lines: ['Acme Corp - internal use only - Page # of #', 'Acme Corp | Travel Policy | Internal']",)

## Purpose
This policy explains how Acme employees book business travel and claim reimbursement for travel costs. It applies to all employees, and to contractors who travel at Acme's request.

## Booking travel
Book flights, trains and hotels through the travel portal at least 14 days before you travel. Economy class is standard for flights under six hours. Flights of six hours or more may be booked in premium economy with your manager's approval.

## Daily limits by city
Hotel and meal costs are reimbursed up to the daily limits below. Limits are per person and include taxes. Use the Other cities row for any city not listed.
City | Hotel per night | Meals per day
London | USD 250 | USD 75
New York | USD 300 | USD 75
Singapore | USD 220 | USD 60
Other cities | USD 

The title, owner and review date came from the PDF's **document properties** (`/Title` plus two custom keys). Many real PDFs have these fields empty, or filled with something unhelpful such as a template name, which is why many teams take descriptive metadata from the document system instead.

Now the second PDF. It *looks* like a policy, and its document properties even say *Visitor Policy*, but the page is a single scanned image:

In [11]:
scanned = parse_pdf(CORPUS_DIR / "scanned-visitor-policy.pdf")
print(f"title from properties: {scanned.title!r}")
print(f"extracted body: {scanned.body!r}")
print(f"notes: {scanned.notes}")

title from properties: 'Visitor Policy'
extracted body: ''
notes: ('1 page, 1 image, 0 characters of text',)


Zero characters. pypdf is not OCR (optical character recognition) software, and no amount of cleaning helps: the text exists only as pixels. The right response is to **detect** it and route the file to OCR, not to index an empty or near-empty record. An image with no text layer is a strong hint; section 5 turns it into a quarantine reason.

### 3.4 Plain text — no metadata slots at all

A `.txt` file has nowhere to put an owner or a date. The adapter takes the first line as the title (a guess, and the validation step will flag what's missing).

In [12]:
def parse_text(path: Path) -> Parsed:
    lines = path.read_text(encoding="utf-8").strip().splitlines()
    title = lines[0].strip() if lines else None               # a guess: plain text has no title field
    return Parsed(title, None, None, "\n".join(lines[1:]).strip(), parser="plain text (standard library)")


PARSERS = {".md": parse_markdown, ".html": parse_html, ".pdf": parse_pdf, ".txt": parse_text}

guide = parse_text(CORPUS_DIR / "manager-pay-review-guide.txt")
print(f"title: {guide.title!r} | owner: {guide.owner} | last_reviewed: {guide.last_reviewed}")

title: 'Manager guide: annual pay review' | owner: None | last_reviewed: None


## 4. Identity and version — two different questions

A RAG system needs to answer two questions about every document, and they need **different** identifiers:

| Question | Identifier | Today | Must stay the same when… |
|---|---|---|---|
| *Which document is this?* | `doc_id` (identity) | the file name without extension, e.g. `travel-policy` | the content is edited (so citations and index entries can be updated, not duplicated) |
| *Which version of it is this?* | `content_hash` (version) | SHA-256 of the normalised body | nothing changed; it **must** change when the text changes |

Common mistakes: using a random UUID per run (nothing is stable), using the content hash as the ID (every edit looks like a brand-new document, and the old one is never deleted), or using a file's modified time as the version (copying, syncing or `git clone` changes it without changing content).

The file name is a readable, reproducible ID **as long as files are never renamed**. Real connectors should prefer the source system's own item ID, which survives renames. Exercise 1 shows what a rename does.

In [13]:
def make_doc_id(path: Path) -> str:
    return path.stem


def sha256(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


a = "Employees receive a home office allowance of USD 500."
b = "Employees receive a home office allowance of USD 600."
print(f"{sha256(a)[:16]}…  {a}")
print(f"{sha256(a)[:16]}…  (same text, same hash: deterministic)")
print(f"{sha256(b)[:16]}…  {b}  <- one character changed")

9d0e7b74370d6210…  Employees receive a home office allowance of USD 500.
9d0e7b74370d6210…  (same text, same hash: deterministic)
e6e17cd1b3d6f8db…  Employees receive a home office allowance of USD 600.  <- one character changed


## 5. Validate and attach permissions — fail closed

Validation decides what happens to each file. Two kinds of findings:

- **Problems** block indexing; the file is **quarantined** with its reasons. Today: no extractable text, no title, or **no permission entry** in the system of record.
- **Warnings** are recorded on the document but don't block it: missing owner, missing or unreadable review date, or a review that is overdue.

Two rules carry most of the security weight:

1. **Permissions come from the system of record, never from the document.** A file's own text can claim anything ("audience: all staff"), and anyone who can edit the file can edit that claim.
2. **Fail closed.** If a file has no permission entry, it is visible to *nobody* until someone fixes the entry. The opposite default (visible to everyone) is how drafts and confidential files leak.

Dates are validated strictly. `04/05/2026` could be 4 May or 5 April, so an unreadable date is recorded as missing with a warning, never guessed.

In [14]:
ACL_FILE = REPO_ROOT / "data" / "sample" / "access-control.json"
ISO_DATE = re.compile(r"\d{4}-\d{2}-\d{2}")


def load_permissions(acl_file: Path) -> dict[str, tuple[str, ...]]:
    export = json.loads(acl_file.read_text(encoding="utf-8"))
    return {path: tuple(sorted(groups)) for path, groups in export["permissions"].items()}


def check_review_date(value: str | None) -> tuple[str | None, list[str]]:
    if not value:
        return None, ["missing last_reviewed"]
    try:
        reviewed = date.fromisoformat(value) if ISO_DATE.fullmatch(value) else None
    except ValueError:
        reviewed = None
    if reviewed is None:
        return None, [f"unreadable last_reviewed {value!r} (expected YYYY-MM-DD; not guessed)"]
    age = (SNAPSHOT_DATE - reviewed).days
    if age > MAX_REVIEW_AGE_DAYS:
        return value, [f"review overdue: last reviewed {age} days before the snapshot"]
    return value, []


def validate(parsed: Parsed, groups: tuple[str, ...] | None) -> tuple[list[str], list[str], str | None]:
    problems, warnings = [], []
    if not parsed.body:
        detail = f" ({'; '.join(parsed.notes)})" if parsed.notes else ""
        problems.append(f"no extractable text{detail}: scanned? run OCR first")
    if not parsed.title:
        problems.append("no title")
    if groups is None:
        problems.append("no entry in the permission export: fail closed (visible to nobody)")
    elif not groups:
        problems.append("permission entry lists no groups: fail closed")
    if not parsed.owner:
        warnings.append("missing owner")
    last_reviewed, date_warnings = check_review_date(parsed.last_reviewed)
    return problems, warnings + date_warnings, last_reviewed


permissions = load_permissions(ACL_FILE)
print(f"{len(permissions)} permission entries in {ACL_FILE.relative_to(REPO_ROOT).as_posix()}\n")
draft = CORPUS_DIR / "draft-reorg-announcement.txt"
print(f"{draft.name} says:  {draft.read_text(encoding='utf-8').splitlines()[2]!r}")
print(f"permission export says: {permissions.get('acme-handbook/' + draft.name)}")

12 permission entries in data/sample/access-control.json

draft-reorg-announcement.txt says:  'Audience: all staff (please share widely)'
permission export says: None


## 6. Run the pipeline

`ingest()` ties the steps together for every file: pick the adapter, parse (a broken file is quarantined, it doesn't crash the run), normalise, look up permissions, validate, then build a `Document` or a `Quarantined` record.

In [15]:
def ingest(root: Path) -> tuple[list[Document], list[Quarantined]]:
    corpus_dir = root / "data" / "sample" / "acme-handbook"
    permissions = load_permissions(root / "data" / "sample" / "access-control.json")
    documents, quarantined, seen = [], [], {}
    for path in list_files(corpus_dir):
        source_path = path.relative_to(root).as_posix()
        adapter = PARSERS.get(path.suffix.lower())
        if adapter is None:
            quarantined.append(Quarantined(source_path, (f"unsupported format {path.suffix!r}",)))
            continue
        try:
            parsed = adapter(path)
        except Exception as error:                            # one bad file must not stop the run
            quarantined.append(Quarantined(source_path, (f"could not parse: {type(error).__name__}: {error}",)))
            continue
        parsed.body = space_headings(normalise(parsed.body))
        groups = permissions.get(path.relative_to(corpus_dir.parent).as_posix())
        problems, warnings, last_reviewed = validate(parsed, groups)
        doc_id = make_doc_id(path)
        if doc_id in seen:
            problems.append(f"duplicate doc_id {doc_id!r}, already used by {seen[doc_id]}")
        if problems:
            quarantined.append(Quarantined(source_path, tuple(problems)))
            continue
        seen[doc_id] = source_path
        documents.append(Document(
            doc_id=doc_id, title=normalise(parsed.title), owner=parsed.owner, last_reviewed=last_reviewed,
            allowed_groups=groups, source_path=source_path, source_format=path.suffix[1:].lower(),
            parser=parsed.parser, content_hash=sha256(parsed.body), body=parsed.body,
            warnings=tuple(warnings),
        ))
    return documents, quarantined


def section_count(body: str) -> int:
    return sum(1 for line in body.splitlines() if line.startswith("## "))


documents, quarantined = ingest(REPO_ROOT)
by_id = {d.doc_id: d for d in documents}

print(f"{len(files)} files -> {len(documents)} documents + {len(quarantined)} quarantined\n")
print(f"{'doc_id':<29}{'fmt':<5}{'sections':>8}{'words':>6}  {'allowed_groups':<22}warnings")
for d in documents:
    print(f"{d.doc_id:<29}{d.source_format:<5}{section_count(d.body):>8}{len(d.body.split()):>6}  "
          f"{', '.join(d.allowed_groups):<22}{'; '.join(d.warnings) or '-'}")

print("\nQUARANTINED (not indexed: fix at the source, then re-run)")
for q in quarantined:
    print(f"  {q.source_path}")
    for reason in q.reasons:
        print(f"      - {reason}")

13 files -> 11 documents + 2 quarantined

doc_id                       fmt  sections words  allowed_groups        warnings
data-retention-policy        md          4   113  all-employees         -
expense-reimbursement-policy md          5   138  all-employees         -
it-software-requests         html        3   145  all-employees         review overdue: last reviewed 470 days before the snapshot
manager-pay-review-guide     txt         0    68  managers, people-team missing owner; missing last_reviewed
new-starter-it-checklist     md          4    92  all-employees         -
parental-leave-policy        md          4   118  all-employees         -
password-and-mfa-standard    md          4   127  all-employees         -
remote-work-policy           md          5   184  all-employees         -
security-incident-reporting  md          4   141  all-employees         -
travel-policy                pdf         5   223  all-employees         -
vpn-access-guide             md          4   

What happened, file by file:

- **8 Markdown policies** passed unchanged, with the same IDs as Day 01, so Day 01's citations still resolve.
- **`travel-policy.pdf`** is indexed with its table rows intact and its header and footer removed.
- **`it-software-requests.html`** is indexed without the navigation and cookie banner, and **warned**: its last review is more than a year old. A stale page can still be the best source, but its owner should know, and an answer could show the date.
- **`manager-pay-review-guide.txt`** is indexed but restricted to `managers` and `people-team`, and warned about its missing owner and date.
- **`scanned-visitor-policy.pdf`** is **quarantined**: no text layer.
- **`draft-reorg-announcement.txt`** is **quarantined**: it says "please share widely", but the permission export has no entry, so it stays invisible.

Here is one complete record, the way later days will receive it:

In [16]:
record = asdict(by_id["travel-policy"])
body = record.pop("body")
print(json.dumps(record, indent=2))
print("\nbody:\n" + body)

{
  "doc_id": "travel-policy",
  "title": "Travel Policy",
  "owner": "Finance",
  "last_reviewed": "2026-04-15",
  "allowed_groups": [
    "all-employees"
  ],
  "source_path": "data/sample/acme-handbook/travel-policy.pdf",
  "source_format": "pdf",
  "parser": "pypdf 6.20.1 (layout mode)",
  "content_hash": "bf4bdf9b29cecca37f3f45b565a210bcda1f096d2a6be1fe5ca44d2ae7a90484",
  "warnings": []
}

body:
## Purpose
This policy explains how Acme employees book business travel and claim reimbursement for travel costs. It applies to all employees, and to contractors who travel at Acme's request.

## Booking travel
Book flights, trains and hotels through the travel portal at least 14 days before you travel. Economy class is standard for flights under six hours. Flights of six hours or more may be booked in premium economy with your manager's approval.

## Daily limits by city
Hotel and meal costs are reimbursed up to the daily limits below. Limits are per person and include taxes. Use the Oth

Finally, write the records to disk. **JSON Lines** (one JSON object per line) is a simple, diffable hand-off format between ingestion and the next stage. The `outputs/` folder is in `.gitignore`, so these files stay on your machine.

In [17]:
OUT_DIR = REPO_ROOT / "outputs" / "day-02"
OUT_DIR.mkdir(parents=True, exist_ok=True)
with (OUT_DIR / "documents.jsonl").open("w", encoding="utf-8", newline="\n") as f:
    for d in documents:
        f.write(json.dumps(asdict(d), ensure_ascii=False) + "\n")
(OUT_DIR / "quarantine.json").write_text(
    json.dumps([asdict(q) for q in quarantined], indent=2), encoding="utf-8", newline="\n")

print(f"wrote {len(documents)} records to {(OUT_DIR / 'documents.jsonl').relative_to(REPO_ROOT).as_posix()}")
print(f"wrote {len(quarantined)} quarantine entries to {(OUT_DIR / 'quarantine.json').relative_to(REPO_ROOT).as_posix()}")

wrote 11 records to outputs/day-02/documents.jsonl
wrote 2 quarantine entries to outputs/day-02/quarantine.json


### Access metadata is not access control (yet)

Recording `allowed_groups` is necessary, but on its own it protects nothing. A preview of what Days 05 and 14 build on top of it:

In [18]:
def visible_to(user_groups: set[str]) -> list[str]:
    return [d.doc_id for d in documents if user_groups & set(d.allowed_groups)]


for user_groups in ({"all-employees"}, {"all-employees", "managers"}):
    print(f"{sorted(user_groups)} -> {len(visible_to(user_groups))} documents")

['all-employees'] -> 10 documents
['all-employees', 'managers'] -> 11 documents


This filter only works if it runs on **every** query, *before* text reaches a prompt, and if the user's groups come from a trusted identity provider rather than from the request. A metadata field nobody filters on is decoration. Day 05 adds filters to a vector index; Day 14 enforces them end to end.

## 7. Did the cleaning matter? Measure it

Each cleaning step should earn its place. These checks compare pypdf's default extraction and the raw HTML text with what the pipeline produced:

In [19]:
def words(text: str) -> set[str]:
    return set(re.findall(r"[^\W_]+", text.lower()))


def on_one_line(text: str, *needles: str) -> bool:
    return any(all(n in line for n in needles) for line in text.splitlines())


raw_pdf = "\n".join(page.extract_text() for page in PdfReader(TRAVEL_PDF).pages)   # pypdf default
raw_html = " ".join(page.dropped + page.lines)                                   # every text node
clean_pdf, clean_html = by_id["travel-policy"].body, by_id["it-software-requests"].body

checks = [
    ("PDF   phrase 'claim reimbursement' is findable", "claim reimbursement" in raw_pdf,
     "claim reimbursement" in clean_pdf),
    ("PDF   header/footer lines in the text", sum("Acme Corp" in line for line in raw_pdf.splitlines()),
     sum("Acme Corp" in line for line in clean_pdf.splitlines())),
    ("PDF   'Singapore' and 'USD 220' on one line", on_one_line(raw_pdf, "Singapore", "USD 220"),
     on_one_line(clean_pdf, "Singapore", "USD 220")),
    ("HTML  cookie-banner text present", "cookies" in words(raw_html), "cookies" in words(clean_html)),
    ("HTML  words of text", len(raw_html.split()), len(clean_html.split())),
]
print(f"{'check':<46}{'raw':>8}{'ingested':>10}")
for name, raw, ingested in checks:
    print(f"{name:<46}{str(raw):>8}{str(ingested):>10}")

check                                              raw  ingested
PDF   phrase 'claim reimbursement' is findable   False      True
PDF   header/footer lines in the text                4         0
PDF   'Singapore' and 'USD 220' on one line      False      True
HTML  cookie-banner text present                  True     False
HTML  words of text                                202       145


And here is the "Daily limits" section that the *same* pipeline produces if you switch pypdf back to its default mode. The cleaning code faithfully joins the column-ordered cells into one long line, and the row structure is lost for good:

In [20]:
plain_body = space_headings(normalise(parse_pdf(TRAVEL_PDF, mode="plain").body))
section = plain_body.split("## Daily limits by city")[1].split("##")[0].strip()
print("## Daily limits by city\n" + section)

## Daily limits by city
Hotel and meal costs are reimbursed up to the daily limits below. Limits are per person and include taxes. Use the Other cities row for any city not listed. City London New York Singapore Other cities Hotel per night USD 250 USD 300 USD 220 USD 180 Meals per day USD 75 USD 75 USD 60 USD 50


Nothing downstream can repair this. A chunker, a retriever and a language model all see the same scrambled sequence. When a table matters, prefer a **structured source** (the spreadsheet or HTML page the PDF was made from), a table-aware extractor, or a human-checked conversion, and add a sanity check like the ones in section 9.

## 8. Freshness — what changed since the last run?

An index is only as current as its last ingestion. Re-processing everything every night is wasteful, and **missing a deletion is dangerous**: a retired policy keeps answering questions. Comparing each run's fingerprints with the previous run tells you exactly what to do:

| Change | How it's detected | What the index must do (Day 05) |
|---|---|---|
| **added** | new `doc_id` | chunk and index it |
| **content changed** | same `doc_id`, new `content_hash` | re-chunk and re-index it (the expensive path) |
| **metadata changed** | same text, different title, owner, date or permissions | update metadata in place; permission changes must propagate fast |
| **removed** | `doc_id` gone | **delete** its chunks |
| **unchanged** | same fingerprints | skip |

The demo below changes a **temporary copy** of the sample data (your files are untouched), runs `ingest()` before and after, and compares.

In [21]:
def fingerprint(doc: Document) -> tuple[str, str]:
    metadata = json.dumps([doc.title, doc.owner, doc.last_reviewed, doc.allowed_groups])
    return doc.content_hash, sha256(metadata)


def diff_runs(before: list[Document], after: list[Document]) -> dict[str, list[str]]:
    old, new = ({d.doc_id: fingerprint(d) for d in run} for run in (before, after))
    changes = {"added": [], "content changed": [], "metadata changed": [], "removed": [], "unchanged": []}
    for doc_id in sorted(old.keys() | new.keys()):
        if doc_id not in old:
            changes["added"].append(doc_id)
        elif doc_id not in new:
            changes["removed"].append(doc_id)
        elif old[doc_id][0] != new[doc_id][0]:
            changes["content changed"].append(doc_id)
        elif old[doc_id][1] != new[doc_id][1]:
            changes["metadata changed"].append(doc_id)
        else:
            changes["unchanged"].append(doc_id)
    return changes


NEW_POLICY = """---
title: Annual Leave Policy
owner: People Team
last_reviewed: 2026-10-01
---

## Entitlement
Full-time employees receive 25 days of paid annual leave per calendar year, plus public holidays.
"""

with tempfile.TemporaryDirectory() as tmp:
    sandbox = Path(tmp)
    shutil.copytree(REPO_ROOT / "data" / "sample", sandbox / "data" / "sample")
    handbook, acl_path = sandbox / "data" / "sample" / "acme-handbook", sandbox / "data" / "sample" / "access-control.json"
    before, _ = ingest(sandbox)

    # 1. An author edits a policy.
    remote = handbook / "remote-work-policy.md"
    remote.write_text(remote.read_text(encoding="utf-8").replace("USD 500", "USD 600"), encoding="utf-8", newline="\n")
    # 2. A policy is retired.
    (handbook / "vpn-access-guide.md").unlink()
    # 3. A new policy is published, and 4. contractors are granted access to an existing one.
    (handbook / "annual-leave-policy.md").write_text(NEW_POLICY, encoding="utf-8", newline="\n")
    acl = json.loads(acl_path.read_text(encoding="utf-8"))
    acl["permissions"]["acme-handbook/annual-leave-policy.md"] = ["all-employees"]
    acl["permissions"]["acme-handbook/password-and-mfa-standard.md"] = ["all-employees", "contractors"]
    acl_path.write_text(json.dumps(acl, indent=2), encoding="utf-8", newline="\n")
    # 5. A file is re-saved with Windows line endings: its bytes change, its text doesn't.
    expense = handbook / "expense-reimbursement-policy.md"
    original_bytes = expense.read_bytes()
    expense.write_bytes(original_bytes.replace(b"\n", b"\r\n"))
    bytes_changed = original_bytes != expense.read_bytes()

    after, after_quarantined = ingest(sandbox)

changes = diff_runs(before, after)
ACTION = {"added": "chunk and index it", "content changed": "re-chunk and re-index",
          "metadata changed": "update metadata in place, no re-chunking", "removed": "delete its chunks from the index",
          "unchanged": "skip"}
print(f"before: {len(before)} documents | after: {len(after)} documents | still quarantined: {len(after_quarantined)}\n")
for kind, ids in changes.items():
    shown = ", ".join(ids) if len(ids) <= 2 else f"{len(ids)} documents"
    print(f"{kind:<17}{shown:<28}-> {ACTION[kind]}")
print(f"\nexpense-reimbursement-policy re-saved with CRLF: file bytes changed = {bytes_changed}, "
      f"classified as {'unchanged' if 'expense-reimbursement-policy' in changes['unchanged'] else 'changed'}")

before: 11 documents | after: 11 documents | still quarantined: 2

added            annual-leave-policy         -> chunk and index it
content changed  remote-work-policy          -> re-chunk and re-index
metadata changed password-and-mfa-standard   -> update metadata in place, no re-chunking
removed          vpn-access-guide            -> delete its chunks from the index
unchanged        8 documents                 -> skip

expense-reimbursement-policy re-saved with CRLF: file bytes changed = True, classified as unchanged


Notice the last line. The file's bytes changed, but the normalised text did not, so the pipeline correctly skips it. Hashing raw bytes, or trusting modified times, would have triggered needless re-indexing.

Notice also what this doesn't catch yet: a document moved or renamed in the source system shows up as one *removed* and one *added*. That's correct but costly, and it breaks old citations. Stable source-system IDs avoid it (exercise 1).

## 9. Sanity checks

These assertions turn the behaviour above into quick, deterministic tests. `tests/test_day_02_notebook.py` runs this notebook in a fresh kernel, so any failing check fails the test. If you change the sample data or settings, expect some to fail: read the message and update deliberately.

In [22]:
# Inventory and outcomes
assert len(files) == 13, len(files)
assert len(documents) == 11 and len(quarantined) == 2, (len(documents), len(quarantined))
assert {q.source_path.rsplit("/", 1)[1] for q in quarantined} == {"draft-reorg-announcement.txt",
                                                                    "scanned-visitor-policy.pdf"}
assert len(by_id) == len(documents), "doc_ids must be unique"

# Day 01 continuity: the Markdown documents keep their Day 01 IDs
assert sorted(d.doc_id for d in documents if d.source_format == "md") == sorted(p.stem for p in CORPUS_DIR.glob("*.md"))
assert len([d for d in documents if d.source_format == "md"]) == 8

# Permissions come from the export, and only from the export
assert all(d.allowed_groups for d in documents)
assert by_id["manager-pay-review-guide"].allowed_groups == ("managers", "people-team")
assert any("fail closed" in r for q in quarantined for r in q.reasons)
assert len(visible_to({"all-employees"})) == 10 and len(visible_to({"all-employees", "managers"})) == 11

# PDF cleaning: table rows, hyphenation, running lines, scanned detection
assert "Singapore | USD 220 | USD 60" in clean_pdf
assert "claim reimbursement" in clean_pdf and "claim reimbursement" not in raw_pdf
assert "Page 1 of 2" not in clean_pdf and "Page 1 of 2" in raw_pdf
assert "Singapore | USD 220" not in plain_body, "default mode should lose the table rows"
assert any("no extractable text" in r for q in quarantined for r in q.reasons)

# HTML cleaning and metadata
assert "Paid licence | Your manager | 3 business days" in clean_html and "cookies" not in words(clean_html)
assert by_id["it-software-requests"].title == "Software requests"
assert any("review overdue" in w for w in by_id["it-software-requests"].warnings)

# Determinism: a second run gives identical records and hashes
assert ingest(REPO_ROOT) == (documents, quarantined)

# Freshness: every kind of change is classified correctly
assert changes == {"added": ["annual-leave-policy"], "content changed": ["remote-work-policy"],
                   "metadata changed": ["password-and-mfa-standard"], "removed": ["vpn-access-guide"],
                   "unchanged": ["data-retention-policy", "expense-reimbursement-policy", "it-software-requests",
                                 "manager-pay-review-guide", "new-starter-it-checklist", "parental-leave-policy",
                                 "security-incident-reporting", "travel-policy"]}, changes
assert bytes_changed

print("All Day 02 sanity checks passed.")

All Day 02 sanity checks passed.


## 10. Exercises

Each takes 5–15 minutes. Restart and rerun the notebook after changing settings. Do the file experiments in a temporary copy (as in section 8) or undo them afterwards, because the sanity checks expect the original sample data.

1. **Rename a file.** In section 8's sandbox, rename `travel-policy.pdf` to `travel-policy-2026.pdf` (and its permission entry). What does `diff_runs` report? What would happen to a Day 01-style citation `[travel-policy#...]`? Sketch how a source-system item ID in the permission export would avoid this.
2. **A date nobody can read.** Change the HTML page's `last-reviewed` to `04/05/2026`. Which warning appears, and why is refusing to guess the right call?
3. **Default PDF mode.** Set `PDF_MODE = "plain"` and rerun. Print `by_id["travel-policy"].body`. Which sanity checks fail, and what would a user asking *"What is the hotel limit in Singapore?"* now depend on?
4. **Turn off one cleaning step.** Comment out `remove_running_lines` in `parse_pdf`. How many times does "Acme Corp" now appear in the travel policy body? Imagine 500 PDFs with the same footer: what would a keyword search for "internal" return?
5. **Who may grant access?** Add an entry for `draft-reorg-announcement.txt` to the permission export. It is now indexed. In a real organisation, who should be allowed to make that change, and how would you audit it?
6. **(Optional, outside the notebook) OCR.** Install an OCR engine such as [Tesseract](https://github.com/tesseract-ocr/tesseract) and run it on the scanned page rendered as an image. What new risks does OCR introduce? (Hint: misread numbers.)

## Summary

- **Ingestion turns many formats into one trustworthy record**: normalised text plus descriptive, access and provenance metadata. Every later day builds on that contract.
- **Extraction is reconstruction.** PDFs store positions, not structure; check tables, headers, hyphens and scanned pages, and measure that your cleaning helps.
- **Fail closed and keep fingerprints.** Files without permissions or text are quarantined with reasons; IDs and content hashes tell you what to add, update or delete next run.

**Next — Day 03: chunking and document structure.** The `## ` headings and table rows produced today become the boundaries we compare when splitting documents into retrievable pieces.